# Camera-ready mechanistic extension — Colab A100

This notebook runs the resumable camera-ready mechanistic sweep for **Qwen3-8B**. It downloads the private input bundle from [`ic-org/inoculate-or-reflect-camera-ready`](https://huggingface.co/datasets/ic-org/inoculate-or-reflect-camera-ready), stages the exact runner, and stores all outputs on Google Drive.

The uploaded bundle includes a valid **6,600-row steering checkpoint**. The runner detects completed condition keys and continues from that checkpoint when this notebook is rerun. The earlier 8,400-row Lambda checkpoint was not available when Lambda was stopped, so this is the checkpoint we can resume here.


## Fixed camera-ready contract

- Model: `Qwen/Qwen3-8B`, seed `42`, 4-bit NF4 QLoRA adapters
- Four arms: untrained base (`arm0`), contaminated baseline (`arm1`), CRT repair (`arm4`), strong IP (`arm6`)
- 200 held-out prompts, five patterns × 40, generated deterministically from the runner
- Six relative depths plus the historical Qwen layers 16 and 18
- Five random-direction controls, a zero-dose parity gate, and activation patching controls
- Expected completion: **13,600 steering rows + 2,800 patching rows**, followed by `manifest.json`

The JSONL files are append-only checkpoints. If Colab disconnects, rerun the setup cells and the run cell; already completed keys are skipped. Gemma is documented below but is not runnable yet because its seed-42 adapter files are not in the private bundle.


In [ ]:
# Install the pinned mechanistic stack. The runner itself uses direct residual hooks;
# NNSight is installed to keep the Colab environment compatible with the canonical Phase 4 setup.
%pip install -q "transformers==5.13.1" "nnsight>=0.7,<0.8" "peft>=0.14" \
    "bitsandbytes>=0.43" "accelerate>=1.2" "huggingface_hub>=0.25" "safetensors"


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import transformers
import peft
import bitsandbytes
import nnsight

print("Python       ", sys.version.split()[0])
print("Torch        ", torch.__version__)
print("Transformers ", transformers.__version__)
print("PEFT         ", peft.__version__)
print("bitsandbytes ", bitsandbytes.__version__)
print("NNSight      ", nnsight.__version__)
print("CUDA         ", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select an A100 runtime before running the mechanistic sweep.")
print("GPU          ", torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_REPO = "ic-org/inoculate-or-reflect-camera-ready"
DRIVE_ROOT = Path("/content/drive/MyDrive/inoculate-or-reflect/camera-ready")
QWEN_OUT = DRIVE_ROOT / "mechanistic" / "qwen3-8b"
RUN_LOG = DRIVE_ROOT / "logs" / "qwen3-8b-mechanistic.log"
BUNDLE_ROOT = Path("/content/ior-camera-ready-bundle")
STAGED_ROOT = Path("/content/ior-camera-ready")
QWEN_OUT.mkdir(parents=True, exist_ok=True)
RUN_LOG.parent.mkdir(parents=True, exist_ok=True)
print("Drive output:", QWEN_OUT)
print("Run log:     ", RUN_LOG)


In [ ]:
# Read the private HF token from Colab Secrets. Never paste a token into this notebook.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise RuntimeError("Add a read-scoped HF_TOKEN in Colab Secrets, then rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir=str(BUNDLE_ROOT),
    allow_patterns=[
        "README.md",
        "artifact_manifest.json",
        "code/*",
        "data/*",
        "qwen3-8b/mechanistic/*",
    ],
)
print("Downloaded bundle:", BUNDLE_ROOT)
print("Bundle files:")
for path in sorted(BUNDLE_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(BUNDLE_ROOT), path.stat().st_size, "bytes")


In [ ]:
# Stage only the files the standalone runner expects under its project-shaped root.
runner_src = BUNDLE_ROOT / "code" / "mechanistic_eval.py"
reserve_src = BUNDLE_ROOT / "data" / "phase4_reserve.jsonl"
runner_dst = STAGED_ROOT / "experiments" / "camera_ready" / "mechanistic_eval.py"
reserve_dst = STAGED_ROOT / "data" / "gcd_sycophancy" / "phase4_reserve.jsonl"
runner_dst.parent.mkdir(parents=True, exist_ok=True)
reserve_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(runner_src, runner_dst)
shutil.copy2(reserve_src, reserve_dst)
print("Runner:", runner_dst)
print("Reserve:", reserve_dst)
print("Runner SHA-256:", hashlib.sha256(runner_dst.read_bytes()).hexdigest())


In [ ]:
# Merge the uploaded checkpoint into Drive without overwriting a longer local run.
# The steering JSONL is the important resume state; the runner regenerates directions deterministically.
bundle_out = BUNDLE_ROOT / "qwen3-8b" / "mechanistic"

def jsonl_count(path):
    if not path.exists():
        return 0
    with path.open() as handle:
        return sum(1 for line in handle if line.strip())

src_steering = bundle_out / "steering_results.jsonl"
dst_steering = QWEN_OUT / "steering_results.jsonl"
source_rows = jsonl_count(src_steering)
target_rows = jsonl_count(dst_steering)
if source_rows > target_rows:
    shutil.copy2(src_steering, dst_steering)
    target_rows = source_rows
    print("Copied checkpoint:", target_rows, "steering rows")
else:
    print("Kept Drive checkpoint:", target_rows, "steering rows")

for name in ("heldout_prompts.jsonl", "directions.npz", "direction_summary.json"):
    src = bundle_out / name
    dst = QWEN_OUT / name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print("Copied", name)
print("Resume directory:", QWEN_OUT)


In [ ]:
# Preflight: validate the checkpoint before loading the 8B model.
EXPECTED_STEERING = 13_600
EXPECTED_PATCH = 2_800
STEERING_KEY = ("analysis", "arm", "direction", "layer", "dose", "sign", "id")
PATCH_KEY = ("analysis", "source", "destination", "layers_key", "amount", "shuffled", "id")

def read_jsonl(path):
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def unique_key_count(path, fields):
    rows = read_jsonl(path)
    keys = {tuple(row.get(field) for field in fields) for row in rows}
    return len(rows), len(keys), rows

heldout_path = QWEN_OUT / "heldout_prompts.jsonl"
reserve_path = STAGED_ROOT / "data" / "gcd_sycophancy" / "phase4_reserve.jsonl"
heldout = read_jsonl(heldout_path)
reserve = read_jsonl(reserve_path)
assert len(heldout) == 200 and len({row["id"] for row in heldout}) == 200
assert len(reserve) == 200 and len({row["id"] for row in reserve}) == 200

steering_path = QWEN_OUT / "steering_results.jsonl"
steering_rows, steering_keys, _ = unique_key_count(steering_path, STEERING_KEY) if steering_path.exists() else (0, 0, [])
assert steering_rows == steering_keys, "Steering checkpoint contains duplicate condition keys"
assert steering_rows <= EXPECTED_STEERING
print({"heldout": len(heldout), "reserve": len(reserve), "steering_rows": steering_rows, "steering_remaining": EXPECTED_STEERING - steering_rows})
print("Preflight passed; the run cell is safe to resume.")


In [ ]:
# Run the camera-ready Qwen sweep. Output is streamed to the notebook and persisted on Drive.
# Rerunning this cell after an interruption is intentional: completed JSONL keys are skipped.
runner = STAGED_ROOT / "experiments" / "camera_ready" / "mechanistic_eval.py"
cmd = [
    sys.executable, "-u", str(runner),
    "--model", "qwen3-8b",
    "--seed", "42",
    "--output-dir", str(QWEN_OUT),
    "--batch-size", "8",
    "--max-new-tokens", "768",
]
print("$", " ".join(cmd))
env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
with RUN_LOG.open("a") as log:
    log.write("\n=== run " + " ".join(cmd) + " ===\n")
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Mechanistic runner exited with code {return_code}; checkpoints remain on Drive.")
print("Runner completed successfully. Log:", RUN_LOG)


In [ ]:
# Post-run validation: require the complete manifest and duplicate-free canonical JSONLs.
manifest_path = QWEN_OUT / "manifest.json"
steering_path = QWEN_OUT / "steering_results.jsonl"
patch_path = QWEN_OUT / "patch_results.jsonl"
assert manifest_path.exists(), "No manifest.json; the run was interrupted and can be resumed."
manifest = json.loads(manifest_path.read_text())
steering_rows, steering_keys, _ = unique_key_count(steering_path, STEERING_KEY)
patch_rows, patch_keys, _ = unique_key_count(patch_path, PATCH_KEY)
assert manifest["status"] == "complete"
assert steering_rows == steering_keys == EXPECTED_STEERING
assert patch_rows == patch_keys == EXPECTED_PATCH
print(json.dumps({
    "status": manifest["status"],
    "model": manifest["model"],
    "steering_rows": steering_rows,
    "patch_rows": patch_rows,
    "primary_layer": manifest["primary_layer"],
    "relative_layers": manifest["relative_layers"],
    "output": str(QWEN_OUT),
}, indent=2))


## Gemma follow-up

The same runner supports `gemma4-12b`, but it requires the seed-42 adapter directories (`contaminated`, `crt_repair`, and `strong_ip`) passed through `--adapter-root`. Those weights are not in the local checkout or the current HF bundle, so no Gemma cell is enabled here. Once the adapter root is uploaded to the private dataset or copied into Drive, the same staging pattern can be extended with:

```bash
python experiments/camera_ready/mechanistic_eval.py \
  --model gemma4-12b --seed 42 \
  --adapter-root /content/drive/MyDrive/.../gemma_seed42 \
  --output-dir /content/drive/MyDrive/inoculate-or-reflect/camera-ready/mechanistic/gemma4-12b \
  --batch-size 4
```

Do not substitute randomly re-trained or different-seed adapters: that would no longer be the planned camera-ready replication.


## After completion

Keep the completed Qwen output directory on Drive. Pull `manifest.json`, `direction_summary.json`, `directions.npz`, `heldout_prompts.jsonl`, `steering_results.jsonl`, and `patch_results.jsonl` back into the repository's `outputs/camera_ready/mechanistic/qwen3-8b/` for local grading and figure regeneration. The large JSONLs can remain on the private HF dataset or Drive; the repository should track only the small canonical summaries required by the paper workflow.
